# Replay OpenAI API Call — PIBot DATA Node

Reproduce la llamada exacta a OpenAI capturada en `logs/run_detail.log`
para la pregunta **"Cual es el valor del imacec del ultimo mes"**.

Permite iterar sobre el system prompt, instrucciones y tools sin necesidad
de ejecutar el flujo completo del grafo.

In [5]:
import subprocess, sys

# Instalar dependencias necesarias en el kernel
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "dateparser", "openai", "python-dotenv", "langgraph",
                       "langchain-core", "psycopg2-binary"])

import os, json
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "qa" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from dotenv import load_dotenv
load_dotenv(ROOT / ".env")

print(f"ROOT: {ROOT}")
print(f"Model: {os.getenv('OPENAI_MODEL', 'gpt-4.1')}")
print(f"Temperature: {os.getenv('DATA_RESPONSE_TEMPERATURE', '0.35')}")

ROOT: /Users/hernanfernandez/Documents/01 Workspace/Notebooks/pibot/bc_pibot
Model: gpt-4o-mini
Temperature: 0.35


## 1. Inputs fijos (capturados de `run_detail.log`)

In [ ]:
QUESTION = "Cual es el valor del imacec del ultimo mes"

ENTITIES = {
    "indicator_ent": "imacec",
    "seasonality_ent": "nsa",
    "frequency_ent": "m",
    "activity_ent": "imacec",
    "region_ent": None,
    "investment_ent": None,
    "price_ent": None,
    "period_ent": ["2026-02-01", "2026-02-28"],
    "calc_mode_cls": "original",
    "activity_cls": "none",
    "activity_cls_resolved": "none",
    "region_cls": "none",
    "investment_cls": "none",
    "req_form_cls": "point",
    "price": "enc",
    "hist": 0,
}

CALC_MODE = ENTITIES["calc_mode_cls"]

print("Question:", QUESTION)
print("Entities:", json.dumps(ENTITIES, indent=2, ensure_ascii=False))

Question: Cual es el valor del imacec del ultimo mes
Entities: {
  "indicator_ent": "imacec",
  "seasonality_ent": "nsa",
  "frequency_ent": "m",
  "activity_ent": "imacec",
  "region_ent": null,
  "investment_ent": null,
  "price_ent": null,
  "period_ent": [
    "2026-02-01",
    "2026-02-28"
  ],
  "calc_mode_cls": "original",
  "activity_cls": "none",
  "activity_cls_resolved": "none",
  "region_cls": "none",
  "investment_cls": "none",
  "req_form_cls": "point",
  "price": "enc",
  "hist": 0
}


## 2. Cargar observations payload desde data_store

In [8]:
from orchestrator.data.catalog_data_search import search_output_payloads

DATA_STORE_DIR = ROOT / "orchestrator" / "memory" / "data_store"

# Mismos kwargs que usa el DATA node para esta clasificación
search_kwargs = {
    "indicator": "imacec",
    "calc_mode": "original",
    "seasonality": "nsa",
    "frequency": "m",
    "price": "enc",
    "has_activity": 0,
    "has_region": 0,
    "has_investment": 0,
    "hist": 0,
}

matches = search_output_payloads(str(DATA_STORE_DIR), **search_kwargs)
print(f"Matches encontrados: {len(matches)}")

observations = matches[0]["payload"]
print(f"Cuadro: {observations.get('cuadro_name')}")
print(f"Series: {len(observations.get('series', []))}")
print(f"Frecuencia: {observations.get('frequency')}")

Matches encontrados: 1
Cuadro: IMACEC, volumen a precios del año anterior encadenado, series empalmadas (promedio 2018=100)
Series: 10
Frecuencia: M


## 3. Construir los mensajes (A–F del log)

In [ ]:
from orchestrator.data.response import (
    _build_initial_messages,
    _build_metric_priority_instruction,
    _build_seasonality_strict_instruction,
    _build_incomplete_period_instruction,
    _build_relative_period_fallback_instruction,
    _build_no_explicit_period_latest_instruction,
    _build_missing_activity_instruction,
    _build_annual_pib_completeness_instruction,
    TOOLS,
    handle_tool_call,
)

# A. System prompt principal (con fecha interpolada)
messages = _build_initial_messages()

# B. Contexto de clasificación
context_payload = {"calc_mode": CALC_MODE or None, "entities": ENTITIES}
messages.append({
    "role": "system",
    "content": (
        "CONTEXTO DE CLASIFICACION (usar para priorizar metricas): "
        + json.dumps(context_payload, ensure_ascii=False)
    ),
})

# C. Metric priority instruction
instr = _build_metric_priority_instruction(CALC_MODE)
if instr:
    messages.append({"role": "system", "content": instr})

# D. Seasonality instruction
instr = _build_seasonality_strict_instruction(
    entities_ctx=ENTITIES, observations=observations
)
if instr:
    messages.append({"role": "system", "content": instr})

# Incomplete period instruction
instr = _build_incomplete_period_instruction(
    question=QUESTION, entities_ctx=ENTITIES, observations=observations
)
if instr:
    messages.append({"role": "system", "content": instr})

# Relative period fallback
instr = _build_relative_period_fallback_instruction(
    question=QUESTION, entities_ctx=ENTITIES, observations=observations
)
if instr:
    messages.append({"role": "system", "content": instr})

# E. No explicit period / latest instruction
instr = _build_no_explicit_period_latest_instruction(
    question=QUESTION, entities_ctx=ENTITIES, observations=observations
)
if instr:
    messages.append({"role": "system", "content": instr})

# Missing activity instruction
instr = _build_missing_activity_instruction(
    entities_ctx=ENTITIES, observations=observations
)
if instr:
    messages.append({"role": "system", "content": instr})

# Annual PIB completeness instruction
instr = _build_annual_pib_completeness_instruction(
    question=QUESTION, entities_ctx=ENTITIES, observations=observations
)
if instr:
    messages.append({"role": "system", "content": instr})

# F. User message
messages.append({"role": "user", "content": QUESTION})

print(f"Total mensajes iniciales: {len(messages)}\n")
for i, m in enumerate(messages):
    role = m['role'].upper()
    content = m['content']
    preview = (content[:120] + '...') if len(content) > 120 else content
    print(f"  {chr(65+i)}. [{role}]: {preview}")

Total mensajes iniciales: 6

  A. [SYSTEM]: Eres un asistente experto en macroeconomía chilena y en interpretación de series del Banco Central de Chile.

FECHA ACTU...
  B. [SYSTEM]: CONTEXTO DE CLASIFICACION (usar para priorizar metricas): {"calc_mode": "original", "entities": {"indicator_ent": "imace...
  C. [SYSTEM]: REGLA ESTRICTA DE REDACCION: en el PRIMER PARRAFO (primera oracion) comienza mencionando el PERIODO analizado (ej: 'En e...
  D. [SYSTEM]: REGLA ESTRICTA DE ESTACIONALIDAD: la serie consultada es no desestacionalizada. No menciones desestacionalización salvo ...
  E. [SYSTEM]: REGLA ESTRICTA DE PERIODO POR DEFECTO: si req_form_cls='latest', o si la pregunta no especifica fecha, Debes responder u...
  F. [USER]: Cual es el valor del imacec del ultimo mes


## 4. Ejecutar llamada OpenAI con function calling loop

In [10]:
import time
from openai import OpenAI

client = OpenAI()
model = os.getenv("OPENAI_MODEL", "gpt-4.1")
temperature = float(os.getenv("DATA_RESPONSE_TEMPERATURE", "0.35"))
max_tool_loops = 16

print(f"Model: {model}")
print(f"Temperature: {temperature}")
print(f"Tools: {[t['function']['name'] for t in TOOLS]}")
print("\n" + "=" * 60)
print("Ejecutando function calling loop...\n")

# Copia de trabajo (se extiende con tool calls)
work_messages = list(messages)
final_response = ""
tool_log = []

t0 = time.perf_counter()

for iteration in range(max_tool_loops):
    response = client.chat.completions.create(
        model=model,
        messages=work_messages,
        tools=TOOLS,
        temperature=temperature,
        stream=False,
    )

    choice = response.choices[0]
    assistant_msg = choice.message

    if assistant_msg.tool_calls:
        work_messages.append(assistant_msg.model_dump())

        for tc in assistant_msg.tool_calls:
            fn_name = tc.function.name
            fn_args = json.loads(tc.function.arguments)
            result = handle_tool_call(fn_name, fn_args, observations)

            tool_log.append({
                "iteration": iteration,
                "tool": fn_name,
                "args": fn_args,
                "result_len": len(result),
                "result_preview": result[:300],
            })
            print(f"  [iter {iteration}] {fn_name}({json.dumps(fn_args)}) -> {len(result)} chars")

            work_messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": result,
            })
    else:
        final_response = assistant_msg.content or ""
        break

elapsed = time.perf_counter() - t0
print(f"\nTiempo total: {elapsed:.2f}s")
print(f"Iteraciones de tool calling: {len(tool_log)}")
print(f"Tokens: prompt={response.usage.prompt_tokens}, "
      f"completion={response.usage.completion_tokens}, "
      f"total={response.usage.total_tokens}")

Model: gpt-4o-mini
Temperature: 0.35
Tools: ['list_series', 'get_series_data', 'rank_series', 'get_extrema', 'get_metadata']

Ejecutando function calling loop...

  [iter 0] get_metadata({}) -> 661 chars
  [iter 1] get_series_data({"series_id": "F032.IMC.IND.Z.Z.EP18.Z.Z.0.M", "frequency": "M", "period": "2026-01"}) -> 448 chars

Tiempo total: 14.47s
Iteraciones de tool calling: 2
Tokens: prompt=6691, completion=68, total=6759


## 5. Respuesta generada

In [11]:
from IPython.display import Markdown, display

display(Markdown(final_response))

En enero de 2026, el IMACEC fue de **111,02** (promedio 2018=100). En comparación con el mismo mes del año anterior, creció un **-0,09%**. Respecto al mes anterior, la variación fue de **-13,51%**.

## 6. Detalle de tool calls ejecutadas

In [12]:
for entry in tool_log:
    print(f"--- Iter {entry['iteration']}: {entry['tool']} ---")
    print(f"Args: {json.dumps(entry['args'], ensure_ascii=False)}")
    print(f"Result ({entry['result_len']} chars): {entry['result_preview']}")
    print()

--- Iter 0: get_metadata ---
Args: {}
Result (661 chars): {"cuadro_name": "IMACEC, volumen a precios del año anterior encadenado, series empalmadas (promedio 2018=100)", "cuadro_id": "imacec_volumen_a_precios_del_ano_anterior_encadenado_series_empalmadas_promedio_2018_100", "classification": {"indicator": "imacec", "calc_mode": ["original", "prev_period", 

--- Iter 1: get_series_data ---
Args: {"series_id": "F032.IMC.IND.Z.Z.EP18.Z.Z.0.M", "frequency": "M", "period": "2026-01"}
Result (448 chars): {"series_id": "F032.IMC.IND.Z.Z.EP18.Z.Z.0.M", "short_title": "Imacec", "frequency": "M", "records": [{"period": "2026-01", "value": 111.020697, "pct": "-13.512755%", "yoy_pct": "-0.089855%", "delta_abs": -17.345858, "yoy_delta_abs": -0.099847, "acceleration_pct": "-23.111716%", "acceleration_yoy": 



## 7. Inspeccionar mensajes completos enviados a OpenAI

In [13]:
for i, m in enumerate(work_messages):
    if isinstance(m, dict):
        role = m.get("role", "?").upper()
        content = str(m.get("content", "") or "(empty)")
        tool_calls = m.get("tool_calls")
    else:
        role = getattr(m, "role", "?").upper()
        content = str(getattr(m, "content", "") or "(empty)")
        tool_calls = getattr(m, "tool_calls", None)

    preview = (content[:200] + "...") if len(content) > 200 else content
    print(f"[{i}] {role}: {preview}")
    if tool_calls:
        for tc in tool_calls:
            if isinstance(tc, dict):
                fn = tc.get("function", {})
                print(f"     -> tool_call: {fn.get('name')}({fn.get('arguments', '')[:80]})")
            else:
                print(f"     -> tool_call: {tc.function.name}({tc.function.arguments[:80]})")
    print()

[0] SYSTEM: Eres un asistente experto en macroeconomía chilena y en interpretación de series del Banco Central de Chile.

FECHA ACTUAL: 2026-03-20
Usa esta fecha para interpretar referencias temporales relativas:...

[1] SYSTEM: CONTEXTO DE CLASIFICACION (usar para priorizar metricas): {"calc_mode": "original", "entities": {"indicator_ent": "imacec", "seasonality_ent": "nsa", "frequency_ent": "m", "activity_ent": "imacec", "r...

[2] SYSTEM: REGLA ESTRICTA DE REDACCION: en el PRIMER PARRAFO (primera oracion) comienza mencionando el PERIODO analizado (ej: 'En el 3er trimestre de 2025, ...') y reporta PRIMERO el valor de 'value' (con su uni...

[3] SYSTEM: REGLA ESTRICTA DE ESTACIONALIDAD: la serie consultada es no desestacionalizada. No menciones desestacionalización salvo que el usuario lo pida explícitamente.

[4] SYSTEM: REGLA ESTRICTA DE PERIODO POR DEFECTO: si req_form_cls='latest', o si la pregunta no especifica fecha, Debes responder usando directamente el ultimo periodo disponib